# Simulación de la Dispersión Gaussiana

Este notebook simula la dispersión bidimensional de un contaminante con una velocidad constante en la dirección **x** e **y**. Utilizamos un modelo gaussiano para describir la concentración de la sustancia en función de su posición y del tiempo.

La ecuación utilizada es:

$$
C(x, y, t) = \frac{Q}{2 \pi \sigma_x \sigma_y} \exp\left(-\frac{(x - u_x t)^2}{2 \sigma_x^2} - \frac{(y-u_y t)^2}{2 \sigma_y^2}\right)
$$

Donde:

- **$C(x, y, t)$** es la concentración de la sustancia en el punto $(x, y)$ y en el tiempo $t$.
- **$Q$** es una constante que representa la fuente.
- **$u_x$** es la velocidad de advección en la dirección $x$ (en este caso, $u_x = 3 \, \text{m/s}$).
- **$u_y$** es la velocidad de advección en la dirección $y$ (en este caso, $u_y = 1 \, \text{m/s}$).
- **$\sigma_x$** y **$\sigma_y$** son los coeficientes de dispersión en las direcciones $x$ e $y$, los cuales crecen con el tiempo a un ritmo determinado por la tasa de crecimiento $k$.
- **$t$** es el tiempo transcurrido desde que la dispersión comenzó.

### Parámetros
En este caso, los parámetros que hemos definido son:
- **$Q = 1.0$**
- **$u_x = 3.0 \, \text{m/s}$**
- **$u_y = 1.0 \, \text{m/s}$**
- **$\sigma_{x0} = 0.1 \, \text{m}$** (dispersión inicial en la dirección $x$)
- **$\sigma_{y0} = 0.1 \, \text{m}$** (dispersión inicial en la dirección $y$)
- **$k = 0.3$** (tasa de crecimiento de la dispersión)

El objetivo es observar cómo la sustancia se dispersa en el tiempo, mientras la "nube" de concentración se desplaza en la dirección $x$ debido a la velocidad constante $u$.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import webbrowser
import os

# Configurar el renderizador si es necesario
pio.renderers.default = 'iframe'

# Función que describe la concentración gaussiana en 2D
def concentracion_gaussiana(x, y, t, u_x=3.0, u_y=1.0, Q=1.0, sigma_x0=0.1, sigma_y0=0.1, k=0.3):
    sigma_x = sigma_x0 + k * t
    sigma_y = sigma_y0 + k * t
    return (Q / (2 * np.pi * sigma_x * sigma_y)) * \
           np.exp(-((x - u_x * t) ** 2 / (2 * sigma_x ** 2)) - ((y - u_y * t) ** 2 / (2 * sigma_y ** 2)))

# Parámetros
x_range = np.linspace(-10, 20, 100)
y_range = np.linspace(-10, 20, 100)
X, Y = np.meshgrid(x_range, y_range)
steps = 100  # Número de pasos en la animación
u_x, u_y = 1.0, 0.5  # Velocidades en los ejes x e y

# Inicialización de la animación
frames = []
for t in range(steps):
    Z = concentracion_gaussiana(X, Y, t * 0.1, u_x, u_y)  # Escalar el tiempo por 0.1 para ajustar la simulación
    frame = go.Frame(
        data=[go.Surface(z=Z, x=X, y=Y, colorscale="Viridis", showscale=False)],
        name=str(t),
        layout=go.Layout(title=f"Simulación de Dispersión Gaussiana en 2D - Tiempo: {t*0.1:.1f} s")
    )
    frames.append(frame)

# Configuración inicial del gráfico
fig = go.Figure(
    data=[go.Surface(z=concentracion_gaussiana(X, Y, 0, u_x, u_y), x=X, y=Y, colorscale="Viridis", showscale=True)],
    layout=go.Layout(
        title="Simulación de Dispersión Gaussiana en 2D - Tiempo: 0.0 s",
        scene=dict(
            zaxis=dict(range=[0, 1], title="Concentración"),
            xaxis=dict(range=[-10, 20], title="x (m)"),
            yaxis=dict(range=[-10, 20], title="y (m)"),
        ),
        updatemenus=[{
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 100, "redraw": True},
                                    "fromcurrent": True, "transition": {"duration": 0}}],
                    "label": "Play",
                    "method": "animate"
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True},
                                      "mode": "immediate", "transition": {"duration": 0}}],
                    "label": "Pause",
                    "method": "animate"
                },
                {
                    "args": [None, {"frame": {"duration": 0, "redraw": True},
                                    "mode": "immediate", "fromcurrent": False,
                                    "transition": {"duration": 0}}],
                    "label": "Reset",
                    "method": "animate"
                }
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top"
        }]
    ),
    frames=frames
)

# Controlar el punto de vista de la cámara
camera = dict(
    eye=dict(x=1.5, y=-1.5, z=4)  # Ajusta la posición de la cámara para ver el gráfico 3D
)

# Aplicar la configuración de la cámara
fig.update_layout(scene_camera=camera)

# Guardar el archivo HTML
output_file = "dispersión_gaussiana.html"
fig.write_html(output_file)

# Abrir el archivo HTML automáticamente en el navegador
webbrowser.open('file://' + os.path.realpath(output_file))


# Modelo Markoviano de Transición para una Distribución Gaussiana 2D

Queremos modelar cómo una nube de probabilidad, inicialmente gaussiana, se desplaza dentro de un mapa 2D. En lugar de mover únicamente el centro de la gaussiana como en un modelo continuo de difusión-advección, usamos un enfoque markoviano sobre una grilla discreta.

Cada celda del mapa representa un estado posible del sistema. La distribución inicial indica la probabilidad de que la nube esté en cada punto del espacio.

En cada paso temporal, la probabilidad se redistribuye hacia las celdas vecinas según una matriz de transición. Esta matriz puede favorecer una dirección manual, por ejemplo:

- Este
- Oeste
- Norte
- Sur
- Noreste
- Noroeste
- Sureste
- Suroeste

La dinámica general es:

$$
P_{t+1}(i,j) = \sum_{m,n} P_t(m,n) \, K((m,n) \rightarrow (i,j))
$$

donde:

- $P_t(i,j)$ es la probabilidad en la celda $(i,j)$ en el tiempo $t$.
- $K$ es el núcleo de transición markoviano.
- La suma de probabilidades se conserva:  
  $$\sum_{i,j} P_t(i,j)=1$$

Este modelo permite representar:

- Una distribución gaussiana inicial.
- Una dirección dominante de desplazamiento.
- Difusión local hacia vecinos cercanos.
- Velocidad de desplazamiento.
- Ruido o dispersión probabilística.
- Mapas de probabilidad externos, obstáculos o zonas preferentes.

# Modelo Markoviano Simple para Movimiento de un Objetivo

El mapa se representa como una grilla 2D.  
Cada celda es un estado posible del objetivo.

En cada paso, el objetivo puede moverse a una celda vecina según una matriz de transición:

\[
K =
\begin{bmatrix}
p_{NO} & p_N & p_{NE} \\
p_O    & p_C & p_E  \\
p_{SO} & p_S & p_{SE}
\end{bmatrix}
\]

La matriz indica la probabilidad de transición hacia cada dirección.

Por ejemplo:

\[
K =
\begin{bmatrix}
0.05 & 0.10 & 0.05 \\
0.10 & 0.20 & 0.30 \\
0.05 & 0.10 & 0.05
\end{bmatrix}
\]

favorece el movimiento hacia el Este, porque \(p_E = 0.30\).

In [ ]:
%matplotlib inline
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Parámetros del mapa
# -----------------------------
nx, ny = 40, 40
steps = 40

x0, y0 = 20, 20  # posición inicial

# -----------------------------
# Matriz de transición 3x3
# -----------------------------
K = np.array([
    [0.05, 0.10, 0.05],
    [0.10, 0.20, 0.30],
    [0.05, 0.10, 0.05]
])

K = K / K.sum()

# -----------------------------
# Distribución inicial
# -----------------------------
P = np.zeros((ny, nx))
P[y0, x0] = 1.0

history = [P.copy()]

# -----------------------------
# Paso markoviano
# -----------------------------
def markov_step(P, K):
    P_next = np.zeros_like(P)

    for i in range(1, P.shape[0] - 1):
        for j in range(1, P.shape[1] - 1):
            P_next[i-1:i+2, j-1:j+2] += P[i, j] * K

    P_next /= P_next.sum()
    return P_next

# -----------------------------
# Simulación
# -----------------------------
for _ in range(steps):
    P = markov_step(P, K)
    history.append(P.copy())

# -----------------------------
# Animación
# -----------------------------
frames = []

for t, Pt in enumerate(history):
    frames.append(
        go.Frame(
            data=[
                go.Heatmap(
                    z=Pt,
                    colorscale="Viridis",
                    zmin=0,
                    zmax=max(h.max() for h in history),
                    colorbar=dict(title="Probabilidad")
                )
            ],
            name=str(t),
            layout=go.Layout(title=f"Transición Markoviana - Paso {t}")
        )
    )

fig = go.Figure(
    data=[
        go.Heatmap(
            z=history[0],
            colorscale="Viridis",
            zmin=0,
            zmax=max(h.max() for h in history),
            colorbar=dict(title="Probabilidad")
        )
    ],
    frames=frames
)

fig.update_layout(
    title="Transición Markoviana desde un punto inicial",
    width=700,
    height=650,
    xaxis_title="x",
    yaxis_title="y",
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 200, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0}
                        }
                    ]
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }
                    ]
                }
            ]
        }
    ]
)
fig.write_html("markov_simple.html", auto_open=True)
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox
from scipy.ndimage import shift, gaussian_filter

# -----------------------------
# 1. Grilla espacial
# -----------------------------
nx, ny = 50, 50
x = np.linspace(-10, 10, nx)
y = np.linspace(-10, 10, ny)
X, Y = np.meshgrid(x, y)

# -----------------------------
# 2. Distribución gaussiana inicial
# -----------------------------
def gaussian_initial(x0=0.0, y0=0.0, sigma_x=1.0, sigma_y=1.0):
    P = np.exp(
        -((X - x0)**2 / (2 * sigma_x**2))
        -((Y - y0)**2 / (2 * sigma_y**2))
    )
    P /= P.sum()
    return P

# -----------------------------
# 3. Dirección de transición
# -----------------------------
directions = {
    "Este": (0, 1),
    "Oeste": (0, -1),
    "Norte": (-1, 0),
    "Sur": (1, 0),
    "Noreste": (-1, 1),
    "Noroeste": (-1, -1),
    "Sureste": (1, 1),
    "Suroeste": (1, -1),
    "Sin dirección": (0, 0)
}

# -----------------------------
# 4. Paso markoviano
# -----------------------------
def markov_step(P, direction="Este", velocity=1.0, diffusion=0.6, persistence=0.8):
    """
    P: matriz de probabilidades actual
    direction: dirección dominante
    velocity: intensidad del desplazamiento
    diffusion: suavizado difusivo
    persistence: peso de la dirección dominante
    """

    dy, dx = directions[direction]

    # Desplazamiento dominante
    P_directed = shift(
        P,
        shift=(dy * velocity, dx * velocity),
        order=1,
        mode="constant",
        cval=0.0
    )

    # Difusión isotrópica local
    P_diffused = gaussian_filter(P, sigma=diffusion)

    # Mezcla markoviana
    P_next = persistence * P_directed + (1 - persistence) * P_diffused

    # Normalización para conservar probabilidad total
    total = P_next.sum()
    if total > 0:
        P_next /= total

    return P_next

# -----------------------------
# 5. Simulación completa
# -----------------------------
def simulate_markov(
    x0=0.0,
    y0=0.0,
    sigma_x0=1.0,
    sigma_y0=1.0,
    direction="Este",
    velocity=1.0,
    diffusion=0.6,
    persistence=0.8,
    steps=20
):
    P = gaussian_initial(x0, y0, sigma_x0, sigma_y0)

    history = [P]

    for _ in range(steps):
        P = markov_step(
            P,
            direction=direction,
            velocity=velocity,
            diffusion=diffusion,
            persistence=persistence
        )
        history.append(P)

    return history

# -----------------------------
# 6. Visualización interactiva
# -----------------------------
def interactive_markov_model(
    x0=0.0,
    y0=0.0,
    sigma_x0=1.0,
    sigma_y0=1.0,
    direction="Este",
    velocity=1.0,
    diffusion=0.6,
    persistence=0.8,
    steps=20,
    surface=False
):
    history = simulate_markov(
        x0=x0,
        y0=y0,
        sigma_x0=sigma_x0,
        sigma_y0=sigma_y0,
        direction=direction,
        velocity=velocity,
        diffusion=diffusion,
        persistence=persistence,
        steps=steps
    )

    frames = []

    for t, P in enumerate(history):
        if surface:
            data = go.Surface(
                x=X,
                y=Y,
                z=P,
                colorscale="Viridis",
                showscale=True
            )
        else:
            data = go.Heatmap(
                x=x,
                y=y,
                z=P,
                colorscale="Viridis",
                zmin=0,
                zmax=max(h.max() for h in history),
                colorbar=dict(title="Probabilidad")
            )

        frames.append(
            go.Frame(
                data=[data],
                name=str(t),
                layout=go.Layout(title=f"Paso markoviano t = {t}")
            )
        )

    initial = frames[0].data[0]

    fig = go.Figure(
        data=[initial],
        frames=frames,
        layout=go.Layout(
            title="Transición Markoviana de una Distribución Gaussiana",
            xaxis=dict(title="x"),
            yaxis=dict(title="y"),
            scene=dict(
                xaxis_title="x",
                yaxis_title="y",
                zaxis_title="Probabilidad"
            ),
            updatemenus=[
                {
                    "type": "buttons",
                    "showactive": False,
                    "buttons": [
                        {
                            "label": "Play",
                            "method": "animate",
                            "args": [
                                None,
                                {
                                    "frame": {"duration": 150, "redraw": True},
                                    "fromcurrent": True,
                                    "transition": {"duration": 0}
                                }
                            ]
                        },
                        {
                            "label": "Pause",
                            "method": "animate",
                            "args": [
                                [None],
                                {
                                    "frame": {"duration": 0, "redraw": True},
                                    "mode": "immediate",
                                    "transition": {"duration": 0}
                                }
                            ]
                        }
                    ]
                }
            ],
            sliders=[
                {
                    "steps": [
                        {
                            "label": str(t),
                            "method": "animate",
                            "args": [
                                [str(t)],
                                {
                                    "frame": {"duration": 0, "redraw": True},
                                    "mode": "immediate",
                                    "transition": {"duration": 0}
                                }
                            ]
                        }
                        for t in range(len(history))
                    ],
                    "currentvalue": {"prefix": "Paso: "}
                }
            ]
        )
    )

    #fig.show()
    fig.write_html("modelo_markov.html", auto_open=True)

# -----------------------------
# 7. Controles interactivos
# -----------------------------
interact(
    interactive_markov_model,
    x0=FloatSlider(value=0.0, min=-8, max=8, step=0.5, description="x inicial"),
    y0=FloatSlider(value=0.0, min=-8, max=8, step=0.5, description="y inicial"),
    sigma_x0=FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="sigma x"),
    sigma_y0=FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="sigma y"),
    direction=Dropdown(options=list(directions.keys()), value="Este", description="Dirección"),
    velocity=FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="Velocidad"),
    diffusion=FloatSlider(value=0.6, min=0.0, max=3.0, step=0.1, description="Difusión"),
    persistence=FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description="Persistencia"),
    steps=IntSlider(value=20, min=1, max=20, step=1, description="Pasos"),
    surface=Checkbox(value=False, description="Vista 3D")
)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import shift, gaussian_filter
from ipywidgets import interact, FloatSlider, IntSlider, SelectMultiple, Checkbox

nx, ny = 50, 50
x = np.linspace(-10, 10, nx)
y = np.linspace(-10, 10, ny)
X, Y = np.meshgrid(x, y)

directions = {
    "Este": (0, 1),
    "Oeste": (0, -1),
    "Norte": (-1, 0),
    "Sur": (1, 0),
    "Noreste": (-1, 1),
    "Noroeste": (-1, -1),
    "Sureste": (1, 1),
    "Suroeste": (1, -1),
}

def gaussian_initial(x0, y0, sigma_x, sigma_y):
    P = np.exp(
        -((X - x0)**2 / (2 * sigma_x**2))
        -((Y - y0)**2 / (2 * sigma_y**2))
    )
    return P / P.sum()

def composed_vector(selected_directions):
    if len(selected_directions) == 0:
        return 0.0, 0.0

    dy_total, dx_total = 0.0, 0.0

    for d in selected_directions:
        dy, dx = directions[d]
        dy_total += dy
        dx_total += dx

    norm = np.sqrt(dx_total**2 + dy_total**2)

    if norm > 0:
        dx_total /= norm
        dy_total /= norm

    return dy_total, dx_total

def markov_step_vector(P, dy, dx, velocity, diffusion, persistence):
    directed = shift(
        P,
        shift=(dy * velocity, dx * velocity),
        order=1,
        mode="constant",
        cval=0.0
    )

    diffused = gaussian_filter(P, sigma=diffusion)

    P_next = persistence * directed + (1 - persistence) * diffused

    if P_next.sum() > 0:
        P_next /= P_next.sum()

    return P_next

def simulate(
    x0=0,
    y0=0,
    sigma_x=1,
    sigma_y=1,
    selected_directions=("Este",),
    velocity=1,
    diffusion=0.6,
    persistence=0.8,
    steps=10,
    show_initial=True
):
    P0 = gaussian_initial(x0, y0, sigma_x, sigma_y)
    P = P0.copy()

    dy, dx = composed_vector(selected_directions)

    history = [P0]

    for _ in range(steps):
        P = markov_step_vector(P, dy, dx, velocity, diffusion, persistence)
        history.append(P)

    if show_initial:
        Z = P0
        title = "Distribución inicial"
    else:
        Z = history[-1]
        title = f"Distribución tras {steps} pasos"

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            x=x,
            y=y,
            z=Z,
            colorscale="Viridis",
            colorbar=dict(title="Probabilidad")
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[x0, x0 + dx * velocity * steps * 0.15],
            y=[y0, y0 - dy * velocity * steps * 0.15],
            mode="lines+markers",
            line=dict(width=4),
            marker=dict(size=8),
            name="Vector compuesto"
        )
    )

    fig.update_layout(
        title=(
            f"{title}<br>"
            f"Direcciones: {list(selected_directions)} | "
            f"Vector resultante dx={dx:.2f}, dy={-dy:.2f}"
        ),
        xaxis_title="x",
        yaxis_title="y",
        width=800,
        height=650
    )

    fig.show()

interact(
    simulate,
    x0=FloatSlider(value=0, min=-8, max=8, step=0.5, description="x inicial"),
    y0=FloatSlider(value=0, min=-8, max=8, step=0.5, description="y inicial"),
    sigma_x=FloatSlider(value=1, min=0.2, max=4, step=0.1, description="sigma x"),
    sigma_y=FloatSlider(value=1, min=0.2, max=4, step=0.1, description="sigma y"),
    selected_directions=SelectMultiple(
        options=list(directions.keys()),
        value=("Este",),
        description="Direcciones"
    ),
    velocity=FloatSlider(value=1, min=0, max=5, step=0.1, description="Velocidad"),
    diffusion=FloatSlider(value=0.6, min=0, max=3, step=0.1, description="Difusión"),
    persistence=FloatSlider(value=0.8, min=0, max=1, step=0.05, description="Persistencia"),
    steps=IntSlider(value=10, min=0, max=40, step=1, description="Pasos"),
    show_initial=Checkbox(value=True, description="Mostrar inicial")
)